# LeetCode 645 — Set Mismatch bằng Python

**Link đề:** https://leetcode.com/problems/set-mismatch/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Bit Manipulation, Sorting

```python
class Solution:
    def findErrorNums(self, nums: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Đọc hiểu bài "một số bị **lặp**, một số bị **mất**" trong dãy `1..n`.
2. Dùng **mảng đếm** (counting array) / `Counter` để tìm cả hai số trong `O(n)`.
3. Dùng **toán học**: tổng `1..n` = `n(n+1)/2` và `set` để suy ra hai số chỉ bằng phép trừ.
4. Biết cách **sắp xếp** rồi quét cặp kề nhau, và cách **đánh dấu âm tại chỗ** (in-place) với bộ nhớ `O(1)`.
5. So sánh 4 lời giải theo Time/Space, nhận ra pattern "giá trị nằm trong `1..n` → dùng giá trị làm chỉ số".

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — chuyện gì đã xảy ra với tập số?

Ban đầu có tập số `1, 2, ..., n`. Do lỗi, **một số bị chép đè thành một số khác** → một số xuất hiện **2 lần**, một số **biến mất**.
Cho mảng `nums` (trạng thái sau lỗi), trả về `[số_lặp, số_mất]`.

**Ví dụ 1:** `nums = [1, 2, 2, 4]` → `[2, 3]` (số 2 lặp, số 3 mất).
**Ví dụ 2:** `nums = [1, 1]` → `[1, 2]` (n = 2, số 1 lặp, số 2 mất).

**Constraints:** `2 <= nums.length <= 10^4`, `1 <= nums[i] <= 10^4`.

**Chú ý quan trọng:**
- `n = len(nums)`; mọi giá trị nằm trong `1..n`.
- Mảng **không nhất thiết đã sắp xếp** (ví dụ `[3, 2, 2]` → lặp 2, mất 1).
- Thứ tự kết quả: **số lặp trước**, số mất sau.

In [1]:
nums = [1, 2, 2, 4]
n = len(nums)
print("n =", n)
print("Tập đúng phải là:", list(range(1, n + 1)))
print("Thực tế có     :", nums)

n = 4
Tập đúng phải là: [1, 2, 3, 4]
Thực tế có     : [1, 2, 2, 4]


## 2. Nền tảng Python cần biết

### 2.1. `range(1, n + 1)` — sinh `1..n`
`range(a, b)` sinh từ `a` đến `b - 1`. Muốn có cả `n` phải viết `range(1, n + 1)`.

In [2]:
n = 4
print(list(range(1, n + 1)))   # [1, 2, 3, 4]
print(list(range(1, n)))       # [1, 2, 3]  -> thiếu n

[1, 2, 3, 4]
[1, 2, 3]


### 2.2. `set` — tập hợp không trùng lặp
- `set(nums)` bỏ các phần tử trùng.
- `x in s` kiểm tra tồn tại nhanh, trung bình `O(1)`.
- `len(set(nums))` = số giá trị **khác nhau**.

In [3]:
nums = [1, 2, 2, 4]
s = set(nums)
print(s, "| len =", len(s))
print(2 in s, 3 in s)
print("sum(nums) =", sum(nums), "| sum(set) =", sum(s))

{1, 2, 4} | len = 3
True False
sum(nums) = 9 | sum(set) = 7


### 2.3. Mảng đếm `count = [0] * (n + 1)`
Khi giá trị nằm trong `1..n`, ta dùng **giá trị làm chỉ số**: `count[v] += 1`.
Tạo dài `n + 1` để chỉ số `n` hợp lệ (bỏ trống ô `0`).

In [4]:
nums = [3, 2, 2]
n = len(nums)
count = [0] * (n + 1)
for v in nums:
    count[v] += 1
print("count =", count)       # [0, 0, 2, 1]
for v in range(1, n + 1):
    print(f"giá trị {v}: xuất hiện {count[v]} lần")

count = [0, 0, 2, 1]
giá trị 1: xuất hiện 0 lần
giá trị 2: xuất hiện 2 lần
giá trị 3: xuất hiện 1 lần


### 2.4. `collections.Counter` — đếm tần suất bằng dict
`Counter(nums)` trả về dict `{giá_trị: số_lần}`. Truy cập key không có → trả `0` (không lỗi như dict thường).

In [5]:
from collections import Counter
c = Counter([1, 2, 2, 4])
print(c)
print("c[2] =", c[2], "| c[3] =", c[3])

Counter({2: 2, 1: 1, 4: 1})
c[2] = 2 | c[3] = 0


### 2.5. Tổng `1 + 2 + ... + n = n(n+1)/2`
Ghép cặp đầu–cuối: `1 + n`, `2 + (n-1)`, ... mỗi cặp bằng `n + 1`, có `n/2` cặp.
Trong Python dùng `n * (n + 1) // 2` để được số nguyên.

In [6]:
for n in [2, 4, 10]:
    print(n, "->", n * (n + 1) // 2, "==", sum(range(1, n + 1)))

2 -> 3 == 3
4 -> 10 == 10
10 -> 55 == 55


## 3. Cách trực giác — Brute Force

"Với mỗi số `v` từ 1 đến n, đếm `nums.count(v)`: nếu 2 lần → số lặp, nếu 0 lần → số mất."

`nums.count(v)` duyệt cả mảng `O(n)`, lặp `n` lần → `O(n²)`. Với `n = 10^4` khoảng `10^8` phép so sánh → chậm.

In [7]:
def find_error_bruteforce(nums: list[int]) -> list[int]:
    n = len(nums)
    dup = missing = -1
    for v in range(1, n + 1):
        c = nums.count(v)          # O(n)
        if c == 2:
            dup = v
        elif c == 0:
            missing = v
    return [dup, missing]

print(find_error_bruteforce([1, 2, 2, 4]))  # [2, 3]
print(find_error_bruteforce([1, 1]))        # [1, 2]
print(find_error_bruteforce([3, 2, 2]))     # [2, 1]

[2, 3]
[1, 2]
[2, 1]


## 4. Nút thắt và insight

**Nút thắt:** brute force đếm lại cả mảng cho từng giá trị — cùng một công việc "đếm" lặp `n` lần.

**Insight 1 (đếm một lần):** duyệt mảng **một lần**, ghi số lần xuất hiện của mỗi giá trị vào `count[v]`. Sau đó quét `1..n`: ô bằng 2 là số lặp, ô bằng 0 là số mất.

**Insight 2 (toán học):** gọi `d` là số lặp, `m` là số mất.
- `sum(nums) = (1 + ... + n) - m + d`
- `sum(set(nums)) = (1 + ... + n) - m` (set bỏ bản sao của `d`)

Suy ra:
```text
d = sum(nums) - sum(set(nums))
m = n(n+1)/2 - sum(set(nums))
```

In [8]:
nums = [1, 2, 2, 4]
n = len(nums)
total = n * (n + 1) // 2
s_all = sum(nums)
s_set = sum(set(nums))
print("tổng 1..n      =", total)
print("sum(nums)      =", s_all)
print("sum(set(nums)) =", s_set)
print("dup     = sum(nums) - sum(set)   =", s_all - s_set)
print("missing = tổng 1..n - sum(set)   =", total - s_set)

tổng 1..n      = 10
sum(nums)      = 9
sum(set(nums)) = 7
dup     = sum(nums) - sum(set)   = 2
missing = tổng 1..n - sum(set)   = 3


## 5. Lời giải chính — Mảng đếm (giảng từng đoạn)

**Đoạn 1:** tạo `count` dài `n + 1`, toàn 0.

**Đoạn 2:** duyệt `nums`, tăng `count[v]`.

**Đoạn 3:** quét `v` từ 1 đến `n`:
- `count[v] == 2` → `dup = v`
- `count[v] == 0` → `missing = v`

**Đoạn 4:** trả `[dup, missing]` (đúng thứ tự đề yêu cầu).

In [9]:
def find_error_counting(nums: list[int]) -> list[int]:
    n = len(nums)
    count = [0] * (n + 1)          # đoạn 1
    for v in nums:                 # đoạn 2
        count[v] += 1
    dup = missing = -1
    for v in range(1, n + 1):      # đoạn 3
        if count[v] == 2:
            dup = v
        elif count[v] == 0:
            missing = v
    return [dup, missing]          # đoạn 4

print(find_error_counting([1, 2, 2, 4]))
print(find_error_counting([1, 1]))
print(find_error_counting([3, 2, 2]))
print(find_error_counting([2, 2]))

[2, 3]
[1, 2]
[2, 1]
[2, 1]


## 6. TRACE từng bước

Ví dụ `nums = [3, 2, 2]` (n = 3).

**Pha đếm:**

| bước | `v` | `count` sau cập nhật |
|---:|---:|---|
| 0 | – | `[0, 0, 0, 0]` |
| 1 | 3 | `[0, 0, 0, 1]` |
| 2 | 2 | `[0, 0, 1, 1]` |
| 3 | 2 | `[0, 0, 2, 1]` |

**Pha quét:**

| `v` | `count[v]` | quyết định |
|---:|---:|---|
| 1 | 0 | `missing = 1` |
| 2 | 2 | `dup = 2` |
| 3 | 1 | bình thường |

Kết quả `[2, 1]`.

In [10]:
def trace_counting(nums: list[int]) -> list[int]:
    n = len(nums)
    count = [0] * (n + 1)
    print("Pha đếm, count ban đầu:", count)
    for i, v in enumerate(nums):
        count[v] += 1
        print(f"  i={i}, v={v} -> count={count}")
    dup = missing = -1
    print("Pha quét:")
    for v in range(1, n + 1):
        note = "bình thường"
        if count[v] == 2:
            dup, note = v, "-> dup"
        elif count[v] == 0:
            missing, note = v, "-> missing"
        print(f"  v={v}, count[v]={count[v]} {note}")
    print("Kết quả:", [dup, missing])
    return [dup, missing]

trace_counting([3, 2, 2])

Pha đếm, count ban đầu: [0, 0, 0, 0]
  i=0, v=3 -> count=[0, 0, 0, 1]
  i=1, v=2 -> count=[0, 0, 1, 1]
  i=2, v=2 -> count=[0, 0, 2, 1]
Pha quét:
  v=1, count[v]=0 -> missing
  v=2, count[v]=2 -> dup
  v=3, count[v]=1 bình thường
Kết quả: [2, 1]


[2, 1]

## 7. Code nộp LeetCode

### 7.1. Cách 1 — Mảng đếm (dễ hiểu nhất, khuyên dùng khi mới học)

In [11]:
class Solution:
    def findErrorNums(self, nums: list[int]) -> list[int]:
        n = len(nums)
        count = [0] * (n + 1)
        for v in nums:
            count[v] += 1
        dup = missing = -1
        for v in range(1, n + 1):
            if count[v] == 2:
                dup = v
            elif count[v] == 0:
                missing = v
        return [dup, missing]

print(Solution().findErrorNums([1, 2, 2, 4]))
print(Solution().findErrorNums([1, 1]))

[2, 3]
[1, 2]


### 7.2. Cách 2 — Toán học với `set` (ngắn nhất)
Dùng 2 công thức ở mục 4. Time `O(n)`, Space `O(n)` cho set.

In [12]:
class SolutionMath:
    def findErrorNums(self, nums: list[int]) -> list[int]:
        n = len(nums)
        s_set = sum(set(nums))
        return [sum(nums) - s_set, n * (n + 1) // 2 - s_set]

print(SolutionMath().findErrorNums([1, 2, 2, 4]))
print(SolutionMath().findErrorNums([3, 2, 2]))

[2, 3]
[2, 1]


### 7.3. Cách 3 — Sắp xếp rồi quét cặp kề nhau

Sau khi sort, số lặp là chỗ `a[i] == a[i-1]`. Số mất là chỗ "nhảy cóc" `a[i] > a[i-1] + 1`.
Hai trường hợp biên của số mất:
- mất số **1**: phần tử đầu `a[0] != 1`.
- mất số **n**: phần tử cuối `a[-1] != n`.

Mẹo gọn: coi như có phần tử ảo `prev = 0` đứng trước, và kiểm tra `a[-1] != n` ở cuối.

In [13]:
def find_error_sorting(nums: list[int]) -> list[int]:
    a = sorted(nums)
    n = len(a)
    dup, missing = -1, -1
    prev = 0                     # phần tử ảo đứng trước, giúp bắt ca mất số 1
    for x in a:
        if x == prev:
            dup = x
        elif x > prev + 1:
            missing = prev + 1
        prev = x
    if a[-1] != n:
        missing = n
    return [dup, missing]

for t in [[1, 2, 2, 4], [1, 1], [3, 2, 2], [2, 2], [1, 3, 3]]:
    print(t, "->", sorted(t), "->", find_error_sorting(t))

[1, 2, 2, 4] -> [1, 2, 2, 4] -> [2, 3]
[1, 1] -> [1, 1] -> [1, 2]
[3, 2, 2] -> [2, 2, 3] -> [2, 1]
[2, 2] -> [2, 2] -> [2, 1]
[1, 3, 3] -> [1, 3, 3] -> [3, 2]


**Kiểm tra hai biên bằng tay:**
- Ca `[2, 2]` (mất 1): `prev = 0`, gặp `x = 2 > 0 + 1` → `missing = 1`. Nhờ phần tử ảo `prev = 0` mà không cần `if` riêng cho số 1.
- Ca `[1, 1]` (mất n = 2): không có chỗ nhảy cóc; kiểm tra cuối `a[-1] = 1 != 2` → `missing = 2`.

In [14]:
class SolutionSort:
    def findErrorNums(self, nums: list[int]) -> list[int]:
        a = sorted(nums)
        n = len(a)
        dup, missing, prev = -1, -1, 0
        for x in a:
            if x == prev:
                dup = x
            elif x > prev + 1:
                missing = prev + 1
            prev = x
        if a[-1] != n:
            missing = n
        return [dup, missing]

print(SolutionSort().findErrorNums([1, 2, 2, 4]))

[2, 3]


### 7.4. Cách 4 — Đánh dấu âm tại chỗ (in-place, O(1) bộ nhớ thêm)

**Ý tưởng:** dùng chính `nums` làm "mảng đánh dấu". Với mỗi giá trị `v`, lật dấu ô `nums[v-1]` thành âm.
- Nếu khi đến lượt `v` mà `nums[v-1]` **đã âm** → `v` đã gặp trước → `v` là số lặp.
- Sau khi duyệt, ô `i` nào **còn dương** → giá trị `i+1` chưa bao giờ xuất hiện → số mất.

Vì các giá trị có thể đã bị lật âm, khi đọc phải dùng `abs(x)`.

⚠ Cách này **thay đổi mảng input** — trong phỏng vấn nên hỏi trước có được phép không (có thể khôi phục lại dấu ở cuối).

In [15]:
def trace_negation(nums: list[int]) -> list[int]:
    nums = nums[:]                      # làm trên bản sao để không phá input của người gọi
    dup = -1
    for i in range(len(nums)):
        v = abs(nums[i])
        if nums[v - 1] < 0:
            dup = v
            print(f"i={i}: v={v}, nums[{v-1}] đã âm -> dup={v} | nums={nums}")
        else:
            nums[v - 1] = -nums[v - 1]
            print(f"i={i}: v={v}, lật nums[{v-1}] thành âm   | nums={nums}")
    missing = -1
    for i, x in enumerate(nums):
        if x > 0:
            missing = i + 1
    print("ô còn dương ->", missing, "| kết quả:", [dup, missing])
    return [dup, missing]

trace_negation([3, 2, 2])

i=0: v=3, lật nums[2] thành âm   | nums=[3, 2, -2]
i=1: v=2, lật nums[1] thành âm   | nums=[3, -2, -2]
i=2: v=2, nums[1] đã âm -> dup=2 | nums=[3, -2, -2]
ô còn dương -> 1 | kết quả: [2, 1]


[2, 1]

In [16]:
class SolutionInPlace:
    def findErrorNums(self, nums: list[int]) -> list[int]:
        dup = missing = -1
        for x in nums:
            v = abs(x)
            if nums[v - 1] < 0:
                dup = v
            else:
                nums[v - 1] = -nums[v - 1]
        for i, x in enumerate(nums):
            if x > 0:
                missing = i + 1
        return [dup, missing]

print(SolutionInPlace().findErrorNums([1, 2, 2, 4]))
print(SolutionInPlace().findErrorNums([3, 2, 2]))

[2, 3]
[2, 1]


**Lưu ý trong `SolutionInPlace`:** vòng `for x in nums` đọc phần tử **hiện tại** của list tại thời điểm duyệt — có thể đã bị lật âm bởi bước trước, nên `abs(x)` là bắt buộc.

## 8. Vì sao đúng?

**Mảng đếm:** sau pha đếm, `count[v]` = số lần `v` xuất hiện. Theo đề, đúng một giá trị xuất hiện 2 lần, đúng một giá trị xuất hiện 0 lần, các giá trị còn lại 1 lần (vì tổng các `count` = `n` và chỉ một số bị đổi). Pha quét gặp đúng 2 ô đặc biệt → trả đúng.

**Toán học:** gọi `T = 1 + ... + n`. Tập giá trị khác nhau trong `nums` = `{1..n} \ {m}` → `sum(set) = T - m`. Mảng `nums` = tập đó cộng thêm một bản sao của `d` → `sum(nums) = T - m + d`. Trừ hai đẳng thức cho ra `d` và `m`.

**Đánh dấu âm:** bất biến — sau khi xử lý `i` phần tử đầu, `nums[v-1] < 0` ⇔ giá trị `v` đã xuất hiện trong `i` phần tử đó. Lần gặp thứ hai của `d` thấy ô đã âm → bắt đúng `d`. Cuối cùng chỉ ô của `m` chưa bị lật.

## 9. Độ phức tạp và so sánh

Gọi `n = len(nums)`.

| Cách | Time | Space thêm | Ưu điểm | Nhược điểm |
|---|---:|---:|---|---|
| Brute `nums.count` | `O(n²)` | `O(1)` | Dễ nghĩ | Chậm |
| Mảng đếm / Counter | `O(n)` | `O(n)` | Rõ ràng, tổng quát | Tốn mảng phụ |
| Toán + `set` | `O(n)` trung bình | `O(n)` | Code 2 dòng | Phải nhớ công thức |
| Sắp xếp | `O(n log n)` | `O(n)` (`sorted`) | Không cần đếm | Chậm hơn, nhiều biên |
| Đánh dấu âm | `O(n)` | `O(1)` | Tối ưu bộ nhớ | Sửa input |

## 10. Bộ test

Nhóm test:
1. Ví dụ đề: `[1,2,2,4]`, `[1,1]`.
2. Mất số **1**: `[2, 2]`, `[3, 2, 2]`.
3. Mất số **n**: `[1, 1]`, `[1, 2, 3, 3]`.
4. Lặp ở cuối/đầu, mảng không sắp xếp.
5. Đối chiếu ngẫu nhiên: tạo `1..n`, chép đè một số → so 5 cách.

In [17]:
import random

cases = [
    ([1, 2, 2, 4], [2, 3]),
    ([1, 1], [1, 2]),
    ([2, 2], [2, 1]),
    ([3, 2, 2], [2, 1]),
    ([1, 2, 3, 3], [3, 4]),
    ([4, 3, 1, 1], [1, 2]),
    ([3, 2, 3, 4, 6, 5], [3, 1]),
]
solvers = [Solution().findErrorNums, SolutionMath().findErrorNums, SolutionSort().findErrorNums,
           SolutionInPlace().findErrorNums, find_error_bruteforce]
for nums, expected in cases:
    for f in solvers:
        got = f(nums[:])           # truyền bản sao vì SolutionInPlace sửa input
        assert got == expected, (f, nums, got, expected)
    print(f"{str(nums):<22} -> {expected} PASS")

[1, 2, 2, 4]           -> [2, 3] PASS
[1, 1]                 -> [1, 2] PASS
[2, 2]                 -> [2, 1] PASS
[3, 2, 2]              -> [2, 1] PASS
[1, 2, 3, 3]           -> [3, 4] PASS
[4, 3, 1, 1]           -> [1, 2] PASS
[3, 2, 3, 4, 6, 5]     -> [3, 1] PASS


In [18]:
rng = random.Random(645)
for _ in range(400):
    n = rng.randint(2, 30)
    arr = list(range(1, n + 1))
    m = rng.randint(1, n)                     # số bị mất
    d = rng.choice([v for v in arr if v != m])  # số bị lặp
    arr[arr.index(m)] = d
    rng.shuffle(arr)
    for f in solvers:
        assert f(arr[:]) == [d, m], (arr, f)
print("400 test ngẫu nhiên: 5 cách cho cùng kết quả PASS")

400 test ngẫu nhiên: 5 cách cho cùng kết quả PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Giả sử mảng đã sắp xếp
Viết `if nums[i] == nums[i+1]` trên mảng gốc → bỏ sót khi bản sao không đứng cạnh nhau, ví dụ `[2, 1, 2]`.

In [19]:
def wrong_assume_sorted(nums):
    dup = -1
    for i in range(len(nums) - 1):
        if nums[i] == nums[i + 1]:
            dup = nums[i]
    return dup

print("sai:", wrong_assume_sorted([2, 1, 2]), "| đúng dup =", Solution().findErrorNums([2, 1, 2])[0])

sai: -1 | đúng dup = 2


### Lỗi 2: Tìm số mất bằng "khoảng nhảy" nhưng quên biên đầu/cuối
Sau khi sort, chỉ xét `a[i] - a[i-1] > 1` sẽ bỏ sót ca mất **1** (`[2, 2]`) hoặc mất **n** (`[1, 1]`).

In [20]:
def wrong_missing_gap(nums):
    a = sorted(nums)
    for i in range(1, len(a)):
        if a[i] - a[i - 1] > 1:
            return a[i - 1] + 1
    return -1

for t in [[1, 2, 2, 4], [2, 2], [1, 1]]:
    print(t, "sai:", wrong_missing_gap(t), "| đúng:", Solution().findErrorNums(t)[1])

[1, 2, 2, 4] sai: 3 | đúng: 3
[2, 2] sai: -1 | đúng: 1
[1, 1] sai: -1 | đúng: 2


### Lỗi 3: Mảng đếm dài `n` thay vì `n + 1`
`count = [0] * n` rồi `count[n] += 1` → `IndexError`. Hoặc dùng `count[v-1]` nhưng quét lại bằng `count[v]` → lệch một.

### Lỗi 4: Trả sai thứ tự `[missing, dup]`
Đề yêu cầu `[số_lặp, số_mất]`.

### Lỗi 5: Quên `abs()` trong cách đánh dấu âm
Giá trị có thể đã bị lật âm → `nums[v - 1]` với `v` âm sẽ truy cập chỉ số âm (Python không báo lỗi mà đọc từ cuối!) → sai âm thầm.

In [21]:
try:
    n = 3
    count = [0] * n
    count[3] += 1
except IndexError as e:
    print("IndexError:", e)

a = [10, 20, 30]
print("Python cho phép chỉ số âm: a[-2] =", a[-2], "-> lỗi logic im lặng nếu quên abs()")

IndexError: list index out of range
Python cho phép chỉ số âm: a[-2] = 20 -> lỗi logic im lặng nếu quên abs()


## 12. Mẹo nhận dạng pattern

- **"Mảng dài n, giá trị trong `1..n`"** → nghĩ ngay tới: mảng đếm, đánh dấu âm `nums[v-1]`, cyclic sort, hoặc công thức tổng.
- **"Một số lặp / một số mất"** → thử phương trình tổng (và tổng bình phương nếu không được dùng set).
- **"Đếm tần suất"** → `Counter` hoặc `count = [0]*(n+1)`.
- Muốn `O(1)` bộ nhớ và được phép sửa input → **dùng dấu của phần tử làm bit đánh dấu**.

Các bài cùng họ: 268 Missing Number, 287 Find the Duplicate Number, 442 Find All Duplicates in an Array, 448 Find All Numbers Disappeared in an Array.

## 13. Biến thể

### 13.1. Không dùng `set`, chỉ dùng tổng và tổng bình phương
Đặt `A = sum(nums) - T = d - m`, `Bq = sum(x²) - sum(i²) = d² - m² = (d - m)(d + m)`.
Suy ra `d + m = Bq / A`, rồi giải hệ. Bộ nhớ `O(1)`, không sửa input.

In [22]:
def find_error_squares(nums: list[int]) -> list[int]:
    n = len(nums)
    diff = sum(nums) - n * (n + 1) // 2                                  # d - m
    sq = sum(x * x for x in nums) - n * (n + 1) * (2 * n + 1) // 6        # d² - m²
    s = sq // diff                                                        # d + m
    d = (diff + s) // 2
    m = s - d
    return [d, m]

for t in [[1, 2, 2, 4], [1, 1], [3, 2, 2], [4, 3, 1, 1]]:
    print(t, "->", find_error_squares(t))

[1, 2, 2, 4] -> [2, 3]
[1, 1] -> [1, 2]
[3, 2, 2] -> [2, 1]
[4, 3, 1, 1] -> [1, 2]


### 13.2. LeetCode 448 — tìm **tất cả** số bị mất
Cùng kỹ thuật đánh dấu âm, chỉ khác là có thể mất nhiều số.

In [23]:
def find_disappeared(nums: list[int]) -> list[int]:
    nums = nums[:]
    for x in nums:
        v = abs(x)
        if nums[v - 1] > 0:
            nums[v - 1] = -nums[v - 1]
    return [i + 1 for i, x in enumerate(nums) if x > 0]

print(find_disappeared([4, 3, 2, 7, 8, 2, 3, 1]))   # [5, 6]

[5, 6]


## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Tính tay:** với `nums = [4, 1, 2, 2]`, tính `sum(nums)`, `sum(set(nums))`, `n(n+1)/2` rồi suy ra kết quả.

**Bài 2 — Trace:** chạy tay cách đánh dấu âm trên `[2, 3, 3, 1]`, ghi `nums` sau mỗi bước.

**Bài 3 — TODO:** viết lời giải dùng `Counter`.

In [24]:
from collections import Counter

def exercise_counter(nums: list[int]) -> list[int]:
    # TODO 1: c = Counter(nums)
    # TODO 2: duyệt v trong 1..n: c[v] == 2 -> dup, c[v] == 0 -> missing
    pass

# assert exercise_counter([1, 2, 2, 4]) == [2, 3]
# assert exercise_counter([2, 2]) == [2, 1]

**Bài 4 — TODO:** viết lời giải chỉ dùng **một `set` `seen`** và tổng: khi thêm phần tử đã có trong `seen` → đó là `dup`; số mất tính từ tổng.

In [25]:
def exercise_seen_set(nums: list[int]) -> list[int]:
    # TODO: seen = set(); duyệt và phát hiện dup; missing = T - (sum(nums) - dup)
    pass

# assert exercise_seen_set([1, 2, 2, 4]) == [2, 3]
# assert exercise_seen_set([3, 2, 2]) == [2, 1]

**Bài 5 — TODO biến thể (LeetCode 442):** trả về **tất cả** số xuất hiện 2 lần (mỗi số xuất hiện 1 hoặc 2 lần, giá trị trong `1..n`), theo thứ tự phát hiện.

In [26]:
def exercise_all_duplicates(nums: list[int]) -> list[int]:
    # TODO: đánh dấu âm; gặp ô đã âm -> thêm vào kết quả
    pass

# assert exercise_all_duplicates([4, 3, 2, 7, 8, 2, 3, 1]) == [2, 3]

**Bài 6 — Mini challenge:** viết lời giải **O(1) bộ nhớ, không sửa input**, dùng XOR. Gợi ý: XOR tất cả `nums` và `1..n` cho ra `d ^ m`; tách theo một bit khác 0 thành hai nhóm.

In [27]:
def exercise_xor(nums: list[int]) -> list[int]:
    # TODO: x = d ^ m; lowbit = x & -x; chia 2 nhóm theo (v & lowbit); xác định số nào là dup
    pass

# assert exercise_xor([1, 2, 2, 4]) == [2, 3]
# assert exercise_xor([3, 2, 2]) == [2, 1]

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Áp dụng 2 công thức mục 4 | `n = 4`, `T = 10` | `9 - 7 = 2`, `10 - 7 = 3` |
| 2 | Lật dấu ô `v-1` | Gặp `3` lần hai thì ô 2 đã âm | Ô còn dương ở chỉ số 3 → mất 4 |
| 3 | `Counter` đếm sẵn | `c[v]` trả 0 nếu thiếu | `for v in range(1, n+1)` |
| 4 | `if x in seen: dup = x` | `missing = T - (sum(nums) - dup)` | `seen.add(x)` sau khi kiểm tra |
| 5 | Giống phát hiện dup ở cách 4 | Không `break` sau lần đầu | `res.append(v)` |
| 6 | `a ^ a = 0`, `a ^ 0 = a` | `x & -x` lấy bit thấp nhất bằng 1 | Sau khi có 2 số `p, q`, số nào có trong `nums` là dup |

## 16. Đáp án tham khảo

**Bài 1:** `sum(nums) = 9`, `sum(set) = 7`, `T = 10` → `dup = 2`, `missing = 3` → `[2, 3]`.

**Bài 2:** `[2,3,3,1]` → v=2: `[2,-3,3,1]` → v=3: `[2,-3,-3,1]` → v=3: ô 2 đã âm → dup=3 → v=1: `[-2,-3,-3,1]`. Ô 3 còn dương → missing=4. Kết quả `[3, 4]`.

In [28]:
print("Bài 1:", SolutionMath().findErrorNums([4, 1, 2, 2]))
print("Bài 2:")
trace_negation([2, 3, 3, 1])

Bài 1: [2, 3]
Bài 2:
i=0: v=2, lật nums[1] thành âm   | nums=[2, -3, 3, 1]
i=1: v=3, lật nums[2] thành âm   | nums=[2, -3, -3, 1]
i=2: v=3, nums[2] đã âm -> dup=3 | nums=[2, -3, -3, 1]
i=3: v=1, lật nums[0] thành âm   | nums=[-2, -3, -3, 1]
ô còn dương -> 4 | kết quả: [3, 4]


[3, 4]

In [29]:
def answer_counter(nums: list[int]) -> list[int]:
    c = Counter(nums)
    n = len(nums)
    dup = missing = -1
    for v in range(1, n + 1):
        if c[v] == 2:
            dup = v
        elif c[v] == 0:
            missing = v
    return [dup, missing]


def answer_seen_set(nums: list[int]) -> list[int]:
    seen = set()
    dup = -1
    for x in nums:
        if x in seen:
            dup = x
        seen.add(x)
    n = len(nums)
    missing = n * (n + 1) // 2 - (sum(nums) - dup)
    return [dup, missing]


def answer_all_duplicates(nums: list[int]) -> list[int]:
    nums = nums[:]
    res = []
    for x in nums:
        v = abs(x)
        if nums[v - 1] < 0:
            res.append(v)
        else:
            nums[v - 1] = -nums[v - 1]
    return res


def answer_xor(nums: list[int]) -> list[int]:
    n = len(nums)
    x = 0
    for v in nums:
        x ^= v
    for v in range(1, n + 1):
        x ^= v                      # x = d ^ m
    lowbit = x & -x
    p = q = 0
    for v in nums:
        if v & lowbit: p ^= v
        else: q ^= v
    for v in range(1, n + 1):
        if v & lowbit: p ^= v
        else: q ^= v
    return [p, q] if p in nums else [q, p]


assert answer_counter([1, 2, 2, 4]) == [2, 3]
assert answer_counter([2, 2]) == [2, 1]
assert answer_seen_set([1, 2, 2, 4]) == [2, 3]
assert answer_seen_set([3, 2, 2]) == [2, 1]
assert answer_all_duplicates([4, 3, 2, 7, 8, 2, 3, 1]) == [2, 3]
assert answer_xor([1, 2, 2, 4]) == [2, 3]
assert answer_xor([3, 2, 2]) == [2, 1]
rng = random.Random(6450)
for _ in range(200):
    n = rng.randint(2, 20)
    arr = list(range(1, n + 1))
    m = rng.randint(1, n); d = rng.choice([v for v in arr if v != m])
    arr[arr.index(m)] = d; rng.shuffle(arr)
    for f in (answer_counter, answer_seen_set, answer_xor):
        assert f(arr) == [d, m]
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findErrorNums(self, nums: list[int]) -> list[int]:
        n = len(nums)
        count = [0] * (n + 1)
        for v in nums:
            count[v] += 1
        dup = missing = -1
        for v in range(1, n + 1):
            if count[v] == 2: dup = v
            elif count[v] == 0: missing = v
        return [dup, missing]
```

Bản 2 dòng nhớ nhanh: `s = sum(set(nums))` → `[sum(nums) - s, n*(n+1)//2 - s]`.

**Checklist:**
- [ ] Trả đúng thứ tự `[dup, missing]`.
- [ ] Không giả sử mảng đã sắp xếp.
- [ ] Đã xét ca mất số `1` và mất số `n`.
- [ ] Mảng đếm dài `n + 1` (hoặc dùng `v - 1` nhất quán).
- [ ] Nếu đánh dấu âm: luôn dùng `abs()` và nhớ là input bị sửa.